# Predicción e Inferencia de Cancelaciones en Tiempo Real

1. **Cargar el predictor robusto de producción** de forma segura (nuestra clase wrapper `BookingPredictor`).
2. **Definir una reserva nueva** con variables en crudo (tal y como entran desde la web del hotel).
3. **Realizar una predicción individual** obteniendo tanto el resultado binario (0 o 1) como la probabilidad exacta de cancelación (0% a 100%) sin preocuparnos por errores de tipo de datos (como pasar cadenas de texto en columnas numéricas).
4. **Realizar predicciones por lotes (Batch)** para múltiples reservas a la vez.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Añadir directorio raíz al path para poder importar src
sys.path.append(str(Path.cwd().parent))

## 1. Inicializar el Predictor Robusto de Producción

En lugar de cargar el archivo `.pkl` directamente (lo cual sería inestable si los datos entrantes contienen errores de tipo o cadenas como `'NULL'`), utilizaremos nuestra clase **`BookingPredictor`** definida en `src/predictor.py`.

Esta clase actúa como un envoltorio (*wrapper*) inteligente que:
1. Carga automáticamente el mejor modelo (`best_model.pkl`).
2. **Limpia y coacciona automáticamente las columnas numéricas**, transformando valores nulos en texto (como `'NULL'`, `'None'` o celdas vacías) a valores `NaN` reales de Python.
3. De esta forma, el imputador del pipeline de Scikit-Learn puede rellenar los nulos con la mediana de entrenamiento en lugar de colapsar con un error.

In [2]:
from src.predictor import BookingPredictor

print("Inicializando el predictor de producción...")
predictor = BookingPredictor()
print("¡Predictor cargado con éxito y protegido contra errores de datos!")

Inicializando el predictor de producción...
¡Predictor cargado con éxito y protegido contra errores de datos!


## 2. Definir una Reserva Individual (Ejemplo de Inferencia)

Simulamos el caso de un cliente que entra a la web de nuestro hotel y realiza una reserva. Creamos un DataFrame de una única fila con las variables en crudo exactas (excepto las que retiramos por Fuga de Datos).

> En `agent` ponemos la cadena `'NULL'`, la cual causaría un error en un pipeline convencional de Scikit-Learn-

In [3]:
# Creamos la reserva simulada en crudo
nueva_reserva = pd.DataFrame(
    [
        {
            "hotel": "City Hotel",
            "lead_time": 145,  # Reserva hecha con casi 5 meses de antelación
            "arrival_date_year": 2016,
            "arrival_date_month": "October",
            "arrival_date_week_number": 42,
            "arrival_date_day_of_month": 12,
            "stays_in_weekend_nights": 2,
            "stays_in_week_nights": 5,
            "adults": 2,
            "children": 0.0,
            "babies": 0,
            "meal": "BB",  # Régimen standard Bed & Breakfast
            "country": "ESP",  # Cliente originario de España
            "market_segment": "Online TA",  # Agencia de viajes Online (ej: Booking)
            "distribution_channel": "TA/TO",
            "is_repeated_guest": 0,
            "previous_cancellations": 0,
            "previous_bookings_not_canceled": 0,
            "reserved_room_type": "A",
            "assigned_room_type": "A",
            "booking_changes": 0,
            "deposit_type": "No Deposit",  # Sin depósito inicial
            "agent": "NULL",  # Cadena de texto 'NULL' que limpiaremos automáticamente
            "days_in_waiting_list": 0,
            "customer_type": "Transient",
            "adr": 110.0,  # Coste de 110€ por noche
            "required_car_parking_spaces": 0,
            "total_of_special_requests": 0,  # Ninguna petición especial
        }
    ]
)

print("Reserva simulada en crudo:")
nueva_reserva

Reserva simulada en crudo:


,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,City Hotel,145,2016,October,42,12,2,5,2,0.0,...,A,A,0,No Deposit,NULL,0,Transient,110.0,0,0


## 3. Realizar la Predicción con la Probabilidad de Cancelación

In [4]:
# Realizar predicción binaria y obtener probabilidad lineal usando el predictor wrapper
prediccion = predictor.predict(nueva_reserva)[0]
probabilidad = predictor.predict_proba(nueva_reserva)[0]

print("--- RESULTADO DE LA INFERENCIA ---")
print(f"-> Predicción de Cancelación (0/1): {prediccion}")
print(f"-> Probabilidad exacta de cancelar: {probabilidad:.2%}")

if prediccion == 1:
    print("\nWARNING: Esta reserva tiene una alta probabilidad de cancelarse.")
    print("Acción recomendada: Enviar correo de reconfirmación o solicitar depósito.")
else:
    print("\nSUCCESS: Esta reserva es segura. El cliente probablemente hará check-in.")

--- RESULTADO DE LA INFERENCIA ---
-> Predicción de Cancelación (0/1): 1
-> Probabilidad exacta de cancelar: 65.10%

Acción recomendada: Enviar correo de reconfirmación o solicitar depósito.


## 4. Predicción por Lotes (Batch Inference)

En producción, los hoteles a menudo quieren predecir el estado de todas las reservas de la semana entrante en un único paso. El pipeline admite múltiples filas de forma nativa sin bucles.

In [5]:
# Creamos un lote de 3 reservas con diferentes niveles de riesgo
lote_reservas = pd.DataFrame(
    [
        {
            # Reserva 1: Alta anticipación y depósito reembolsable (Bajo riesgo)
            "hotel": "Resort Hotel",
            "lead_time": 10,
            "arrival_date_year": 2017,
            "arrival_date_month": "May",
            "arrival_date_week_number": 20,
            "arrival_date_day_of_month": 15,
            "stays_in_weekend_nights": 1,
            "stays_in_week_nights": 2,
            "adults": 2,
            "children": 0.0,
            "babies": 0,
            "meal": "BB",
            "country": "PRT",
            "market_segment": "Direct",
            "distribution_channel": "Direct",
            "is_repeated_guest": 1,
            "previous_cancellations": 0,
            "previous_bookings_not_canceled": 5,
            "reserved_room_type": "A",
            "assigned_room_type": "A",
            "booking_changes": 2,
            "deposit_type": "No Deposit",
            "agent": "NULL",
            "days_in_waiting_list": 0,
            "customer_type": "Transient",
            "adr": 85.0,
            "required_car_parking_spaces": 1,
            "total_of_special_requests": 2,
        },
        {
            # Reserva 2: Historial de cancelaciones previas y reserva No Reembolsable (Riesgo Extremo)
            "hotel": "City Hotel",
            "lead_time": 320,
            "arrival_date_year": 2016,
            "arrival_date_month": "September",
            "arrival_date_week_number": 38,
            "arrival_date_day_of_month": 20,
            "stays_in_weekend_nights": 0,
            "stays_in_week_nights": 3,
            "adults": 2,
            "children": 0.0,
            "babies": 0,
            "meal": "BB",
            "country": "PRT",
            "market_segment": "Groups",
            "distribution_channel": "TA/TO",
            "is_repeated_guest": 0,
            "previous_cancellations": 2,
            "previous_bookings_not_canceled": 0,
            "reserved_room_type": "A",
            "assigned_room_type": "A",
            "booking_changes": 0,
            "deposit_type": "Non Refund",
            "agent": "1",
            "days_in_waiting_list": 0,
            "customer_type": "Transient",
            "adr": 70.0,
            "required_car_parking_spaces": 0,
            "total_of_special_requests": 0,
        },
        {
            # Reserva 3: Cliente corporativo, baja anticipación (Bajo riesgo)
            "hotel": "City Hotel",
            "lead_time": 5,
            "arrival_date_year": 2017,
            "arrival_date_month": "June",
            "arrival_date_week_number": 25,
            "arrival_date_day_of_month": 21,
            "stays_in_weekend_nights": 0,
            "stays_in_week_nights": 1,
            "adults": 1,
            "children": 0.0,
            "babies": 0,
            "meal": "BB",
            "country": "GBR",
            "market_segment": "Corporate",
            "distribution_channel": "Corporate",
            "is_repeated_guest": 1,
            "previous_cancellations": 0,
            "previous_bookings_not_canceled": 12,
            "reserved_room_type": "A",
            "assigned_room_type": "A",
            "booking_changes": 1,
            "deposit_type": "No Deposit",
            "agent": "NULL",
            "days_in_waiting_list": 0,
            "customer_type": "Transient",
            "adr": 95.0,
            "required_car_parking_spaces": 0,
            "total_of_special_requests": 1,
        },
    ]
)

print("Lote de 3 reservas cargado con éxito.")

Lote de 3 reservas cargado con éxito.


In [6]:
# Realizar predicciones y obtener probabilidades sobre el lote entero usando el predictor wrapper
predicciones_lote = predictor.predict(lote_reservas)
probabilidades_lote = predictor.predict_proba(lote_reservas)

# Añadir resultados al DataFrame para mostrarlos en una sola tabla estática
resultados_df = lote_reservas[
    ["hotel", "lead_time", "deposit_type", "previous_cancellations"]
].copy()
resultados_df["Predicción (is_canceled)"] = predicciones_lote
resultados_df["Probabilidad de Cancelar"] = [f"{prob:.2%}" for prob in probabilidades_lote]

print("\n--- RESULTADOS DE PREDICCIÓN POR LOTES ---")
resultados_df


--- RESULTADOS DE PREDICCIÓN POR LOTES ---


,hotel,lead_time,deposit_type,previous_cancellations,Predicción (is_canceled),Probabilidad de Cancelar
0,Resort Hotel,10,No Deposit,0,0,0.05%
1,City Hotel,320,Non Refund,2,1,99.97%
2,City Hotel,5,No Deposit,0,0,0.81%
